In [1]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('token')  # safely fetch from secrets

if hf_token:
    login(hf_token)
    print("Successfully logged in to Hugging Face!")
else:
    print("Hugging Face token not found. Add it in Colab Secrets.")

Successfully logged in to Hugging Face!


In [2]:
# Install specific compatible versions
!pip install numpy==1.19.5 --quiet
!pip install git+https://github.com/AIPHES/emnlp19-moverscore.git --quiet
!pip install accelerate rouge-score bert-score nltk torch torchvision pyemd --quiet
!pip uninstall torchao -y
!pip install "datasets<4.0.0"

# Verify
try:
    from moverscore_v2 import get_idf_dict, word_mover_score
    print("✓ MoverScore installed successfully")
except Exception as e:
    print(f"✗ Error: {e}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 52.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
ERROR: Exception:
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/commands/install.py", line 377, in run
    requirement_set = resolver.resolve(
        reqs, check_supported_wheels=not options.target_dir
    )
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 95, in resolve
    result = self._result = resolver.resolve(
                            ~~~~~~~~~~~~~~~~^
        collected.requirements, max_rounds=limit_how_complex_resolution_can_b

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ MoverScore installed successfully


In [3]:
import nltk; nltk.download('wordnet', quiet=True); nltk.download('omw-1.4', quiet=True)

import torch
import gc
import math
import os
import threading
from typing import Optional
from PIL import Image

# Patch for numpy.float deprecation in MoverScore
import numpy as np
if not hasattr(np, 'float'):
    np.float = float

import re
import numpy as np
from transformers import EvalPrediction
import nltk
nltk.download('punkt', quiet=True)

from datasets import load_dataset
from transformers import (
    Gemma3ForConditionalGeneration,
    AutoProcessor,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from moverscore_v2 import get_idf_dict, word_mover_score
from peft import LoraConfig, get_peft_model


def load_image_from_source(image_source):
    """
    Robust image loading for RefCOCO examples.

    lmms-lab-encoder/RefCOCO's `image` column is already a decoded
    PIL.Image (the raw COCO image, with NO region visualization drawn on
    it — see draw_region_indicator() below for that), so this is mostly a
    defensive convert-to-RGB helper.
    """
    try:
        if isinstance(image_source, Image.Image):
            return image_source.convert('RGB') if image_source.mode != 'RGB' else image_source

        # Fallback in case a path/bytes slips through instead of a PIL Image.
        if isinstance(image_source, str) and os.path.exists(image_source):
            image = Image.open(image_source)
            return image.convert('RGB') if image.mode != 'RGB' else image

        if isinstance(image_source, bytes):
            from io import BytesIO
            image = Image.open(BytesIO(image_source))
            return image.convert('RGB') if image.mode != 'RGB' else image

        return None

    except Exception as e:
        print(f"Failed to load image: {e}")
        return None


def draw_region_indicator(image, bbox, color=(255, 0, 0), width_frac=0.006):
    """
    Draw a red RECTANGLE around the referred region.

    FIX: this used to draw a padded ellipse and the prompt referred to
    "the area circled in the image". That doesn't match how this dataset
    is actually meant to be used: lmms-eval's own reference implementation
    for this exact dataset (lmms_eval/tasks/refcoco/utils.py,
    refcoco_bbox_doc_to_visual) draws a plain rectangle at the *exact*
    bbox, no padding, no circle:

        bbox_xy = [bbox[0], bbox[1], bbox[0] + bbox[2], bbox[1] + bbox[3]]
        draw.rectangle(bbox_xy, outline="red")

    We mirror that (bbox is COCO-style [x, y, w, h] in pixel coordinates,
    confirmed against the dataset's own schema) so the visual indicator
    matches what the harness/paper convention actually is, and so the
    prompt's wording ("the region" / "the box", not "circled") matches
    what's drawn. We keep a slightly thicker line than the harness's
    default 1px, since a 1px line is prone to disappearing once the image
    gets downsampled by the vision processor before the model ever sees it
    — the harness only ever runs zero-shot inference on the original-size
    image, so it doesn't have to worry about that, but we do.
    """
    from PIL import ImageDraw

    annotated = image.copy()
    if not bbox or len(bbox) != 4:
        return annotated

    x, y, w, h = bbox
    if w <= 0 or h <= 0:
        return annotated

    # Origin format (top x, top y, width, height) -> (x0, y0, x1, y1),
    # exactly as lmms-eval's refcoco_bbox_doc_to_visual does it. No padding.
    x0, y0 = x, y
    x1 = min(annotated.width, x + w)
    y1 = min(annotated.height, y + h)

    line_width = max(2, int(round(min(annotated.width, annotated.height) * width_frac)))

    draw = ImageDraw.Draw(annotated)
    draw.rectangle([x0, y0, x1, y1], outline=color, width=line_width)
    return annotated


def extract_raw_answers(example):
    """
    Return the list of raw human reference-expression strings for a
    RefCOCO example, for use with the soft-accuracy metric.

    Unlike VQAv2's dict-based `answers` field, RefCOCO's `answer` field is
    already a flat list of plain strings (the 2-4 human referring
    expressions collected for that region).
    """
    answers_field = example.get('answer')
    if not answers_field:
        return []
    raw = []
    for a in answers_field:
        ans = a.get('answer') if isinstance(a, dict) else a
        if ans:
            raw.append(str(ans))
    return raw


class LoRAConfig:
    """Configuration for LoRA-only training on Gemma 3-4B-IT models."""

    def __init__(self):
        # Gemma 3-4B-IT model ID
        self.model_id = "google/gemma-3-4b-it"
        self.batch_size = 1
        # FIX: with batch_size=1, grad_accum=1 means every step is a
        # full-strength update from a single noisy example. Accumulating
        # over 4 examples smooths that out and helps avoid the "EOS
        # collapse" failure mode (model learns to emit EOS immediately,
        # since EOS is the easiest/most common target token in a batch
        # of very short captions).
        self.gradient_accumulation_steps = 4
        self.num_epochs = 3

        # FIX: 1e-5 was too small for a LoRA adapter, but 2e-4 combined
        # with batch_size=1 / no accumulation turned out to be too
        # aggressive for this tiny (32-example) dataset — it drove the
        # model into "EOS collapse" (predicting EOS as the very first
        # token, since EOS is the easiest/most common target across such
        # short captions) within a single epoch. 5e-5, combined with the
        # gradient accumulation above, is a middle ground: still large
        # enough to move a zero-initialized LoRA adapter, gentle enough
        # not to immediately overfit to the cheapest shortcut.
        self.learning_rate = 5e-5

        self.precision = "bf16"
        self.max_length = 1024

        # LoRA configuration
        self.lora_rank = 8
        self.lora_alpha = 16
        self.lora_dropout = 0.05

        # Gemma 3 target modules
        # FIX: added "o_proj" — adapting q/k/v but not the attention output
        # projection is unusual and limits how much the attention block can
        # actually change; o_proj is cheap to add and standard practice.
        self.lora_target_modules = [
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
        ]

        # Vision encoder LoRA
        self.apply_lora_to_vision = True
        self.vision_lora_modules = [
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
        ]

        # Dataset configuration
        # lmms-lab-encoder/RefCOCO is parquet-native (no trust_remote_code
        # needed). It's a repackaged *evaluation-only* version of RefCOCO:
        # it ships only val/test/testA/testB — there is no "train" split.
        # All four splits carry real reference-expression answers (unlike
        # VQAv2's blind test), so we don't need any held-out-from-val
        # workaround here. We repurpose "val" (the largest split) as our
        # train source, and testA/testB as validation/test.
        self.dataset_name = "lmms-lab-encoder/RefCOCO"
        self.num_workers = 0
        self.pin_memory = True
        self.drop_last = True

        self.answer_max_length = 512
        self.ignore_index = -100

        # FIX: cap generation length during eval. Reference captions are a
        # few words; letting the (still-learning) model ramble for up to
        # 512 tokens on every eval example wastes huge amounts of eval time
        # for no benefit. Once training is fixed the model should also
        # learn to emit EOS quickly, but this cap keeps eval fast either way.
        self.generation_max_new_tokens = 64


def main():
    config = LoRAConfig()

    print("Loading processor and model for Gemma 3-4B-IT (RefCOCO)...")
    processor = AutoProcessor.from_pretrained(
        config.model_id,
        token=hf_token,
        trust_remote_code=True,
        padding_side="left",
    )

    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

    model = Gemma3ForConditionalGeneration.from_pretrained(
        config.model_id,
        device_map="auto",
        token=hf_token,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    )

    # Apply LoRA configuration
    target_modules = config.lora_target_modules.copy()

    lora_config = LoraConfig(
        r=config.lora_rank,
        lora_alpha=config.lora_alpha,
        target_modules=target_modules,
        lora_dropout=config.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        inference_mode=False,
        use_rslora=True,
    )

    model = get_peft_model(model, lora_config)

    # Count vision and language modules
    from peft.tuners.lora import LoraLayer
    vision_modules = 0
    language_modules = 0

    for name, module in model.named_modules():
        if isinstance(module, LoraLayer) and module.r.get('default', 0) > 0:
            name_lower = name.lower()
            is_vision = any([
                'vision_tower' in name_lower,
                'vision_model' in name_lower,
                'vision_encoder' in name_lower,
                'visual' in name_lower and 'language' not in name_lower,
                name.startswith('model.vision'),
                name.startswith('vision_tower'),
            ])

            if is_vision:
                vision_modules += 1
            else:
                language_modules += 1

    print(f"Instrumented {vision_modules} vision modules + {language_modules} language modules")
    print(f"Total LoRA modules: {vision_modules + language_modules}")

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    print(f"Trainable params: {trainable_params:,}")
    print(f"Total params: {total_params:,}")
    print(f"Percentage: {100 * trainable_params / total_params:.3f}%")

    model.enable_input_require_grads()
    model.config.use_cache = False

    print("Loading RefCOCO dataset with splits...")
    # Parquet-native, so no trust_remote_code / streaming workaround
    # needed — this dataset is small (17.6K rows total).
    full_dataset = load_dataset(config.dataset_name)

    # NOTE: this repackaged dataset has no "train" split — only
    # val/test/testA/testB, all with real reference-expression answers.
    # We repurpose "val" (largest, 8.8K) as our train source, and
    # testA/testB as validation/test.
    train_dataset = full_dataset["val"].shuffle(seed=42).select(range(32))
    val_dataset = full_dataset["testA"].shuffle(seed=42).select(range(4))
    test_dataset = full_dataset["testB"].shuffle(seed=42).select(range(4))

    print(f"Final splits → Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

    def tokenize_vqa(example):
        """
        Process RefCOCO examples: draw the referred region, then build a
        real (prompt + answer) training sequence.

        FIX (root cause of the original bug): the previous version
        tokenized *only* the prompt padded to max_length, then pasted the
        answer token ids into the tail of a `labels` array without ever
        putting them into `input_ids`. Because the processor pads on the
        left, that tail is where the literal prompt/template tokens live
        (e.g. "<start_of_turn>model\\n"), so the model was being trained to
        predict answer tokens as if they followed prompt tokens they never
        actually preceded — a nonsensical target that produced huge loss
        and no real learning signal.

        This version builds one real sequence: [prompt_tokens][answer_tokens][eos],
        and masks the labels for the prompt portion with -100 so loss is
        only computed on the actual answer + eos.
        """
        try:
            if 'image' not in example or example['image'] is None:
                return None

            image = load_image_from_source(example['image'])

            if image is None:
                return None

            if not isinstance(image, Image.Image):
                return None
            if image.mode != 'RGB':
                image = image.convert('RGB')

            # RefCOCO's raw image has no visualization baked in — draw the
            # referred region as a red rectangle using the ground-truth bbox.
            bbox = example.get('bbox')
            image = draw_region_indicator(image, bbox)

            # FIX: this used to read example.get('question') and only fall
            # back to a hardcoded string if it was empty — i.e. it trusted
            # the dataset's own `question` column as the primary prompt.
            # lmms-eval's own task definition for this exact dataset
            # (lmms_eval/tasks/refcoco/utils.py, refcoco_doc_to_text)
            # explicitly does NOT do this — it comments out `doc["question"]`
            # entirely and hardcodes:
            #
            #     return "Provide a short description for this region."
            #
            # That's a strong signal the `question` column in this
            # repackaged dataset isn't a reliable, ready-to-use instruction
            # for the region-captioning direction of the task, so we match
            # the harness's own choice here instead of the dataset column.
            question = "Provide a short description for this region."

            # FIX: the base instruct model's default habit is to write a
            # full explanatory paragraph ("Here's a description of this
            # region: **\"...\"** Here's why..."), but the references are
            # short (1-4 word) noun phrases. Leaving the model to unlearn
            # that habit purely from a small LoRA adapter's loss signal is
            # slow; making the brevity requirement explicit in the prompt
            # itself lets it lean on instruction-following it already has,
            # which should help ROUGE-L/METEOR (both penalize length
            # mismatch heavily) well before the adapter alone would fix it.
            # This one sentence is our own addition on top of the harness's
            # base instruction (the harness only ever does zero-shot
            # inference and doesn't need this; we're fine-tuning a small
            # adapter on 32 examples and do). Applied uniformly to train
            # and eval since both share this function/prompt.
            question = f"{question} Respond with only the short description itself (a few words), with no explanation or extra text."

            # `answer` is a list of 2-4 human referring expressions for the
            # region; train against the first one (raw list is kept
            # separately for multi-reference eval metrics and the
            # soft-accuracy metric via extract_raw_answers).
            answers_list = example.get('answer') or []
            answer = (answers_list[0] if answers_list else '').strip()

            if not question or not answer:
                return None

            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image"},
                        {"type": "text", "text": question}
                    ]
                }
            ]

            text_prompt = processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            # Tokenize the PROMPT (with image) alone, no padding here.
            # We deliberately don't pass max_length/truncation on the
            # prompt: the prompt (including image tokens) must never be
            # truncated, or the model loses the image/instruction it needs.
            prompt_inputs = processor(
                text=[text_prompt],
                images=[image],
                return_tensors="pt",
                padding=False,
            )

            prompt_ids = prompt_inputs['input_ids'].squeeze(0)
            prompt_mask = prompt_inputs['attention_mask'].squeeze(0)
            token_type_ids = prompt_inputs.get('token_type_ids', None)
            if token_type_ids is not None:
                token_type_ids = token_type_ids.squeeze(0)

            if not isinstance(prompt_ids, torch.Tensor):
                prompt_ids = torch.tensor(prompt_ids, dtype=torch.long)
            if not isinstance(prompt_mask, torch.Tensor):
                prompt_mask = torch.tensor(prompt_mask, dtype=torch.long)
            if token_type_ids is not None and not isinstance(token_type_ids, torch.Tensor):
                token_type_ids = torch.tensor(token_type_ids, dtype=torch.long)

            if prompt_ids.shape[0] >= config.max_length:
                # Prompt alone (usually dominated by image tokens) already
                # doesn't leave room for any answer — skip this example
                # rather than truncating the image context.
                return None

            eos_id = processor.tokenizer.eos_token_id
            answer_tokens = processor.tokenizer(
                answer,
                add_special_tokens=False,
                return_tensors="pt"
            )['input_ids'].squeeze(0)
            if not isinstance(answer_tokens, torch.Tensor):
                answer_tokens = torch.tensor(answer_tokens, dtype=torch.long)

            # FIX: append EOS to the training target so the model learns to
            # stop generating instead of rambling indefinitely.
            eos_tensor = torch.tensor([eos_id], dtype=answer_tokens.dtype)
            answer_tokens = torch.cat([answer_tokens, eos_tensor], dim=0)

            # If prompt + answer would exceed max_length, truncate the
            # ANSWER (never the prompt/image) to fit.
            remaining_budget = config.max_length - prompt_ids.shape[0]
            if remaining_budget <= 0:
                return None
            if answer_tokens.shape[0] > remaining_budget:
                # keep room for the eos token we appended
                answer_tokens = torch.cat(
                    [answer_tokens[: remaining_budget - 1], eos_tensor], dim=0
                )

            # Build the real training sequence: prompt followed by answer.
            input_ids = torch.cat([prompt_ids, answer_tokens], dim=0)
            attention_mask = torch.cat(
                [prompt_mask, torch.ones_like(answer_tokens)], dim=0
            )
            labels = torch.cat(
                [
                    torch.full_like(prompt_ids, config.ignore_index),
                    answer_tokens.clone(),
                ],
                dim=0,
            )

            if token_type_ids is not None:
                token_type_ids = torch.cat(
                    [token_type_ids, torch.zeros_like(answer_tokens)], dim=0
                )

            output = {
                'input_ids': input_ids,
                'attention_mask': attention_mask,
                'labels': labels,
                # FIX: the length of the prompt-only portion of input_ids
                # (before the answer+eos were appended). Training needs the
                # full [prompt][answer] sequence for teacher forcing, but
                # at EVAL time model.generate() must only ever see the
                # prompt — otherwise the model is handed the ground-truth
                # caption as context and asked to "continue" past it,
                # which produces meaningless (often empty, special-token)
                # output instead of an actual generated caption.
                # LoRATrainer.prediction_step uses this to reconstruct a
                # prompt-only input before calling generate().
                'prompt_length': int(prompt_ids.shape[0]),
                # Raw (un-normalized) list of the ~2-4 human answers, kept
                # around purely for the official VQA soft-accuracy metric
                # at eval time. Not used in training/loss.
                'vqa_answers': extract_raw_answers(example),
            }
            if token_type_ids is not None:
                output['token_type_ids'] = token_type_ids

            if 'pixel_values' in prompt_inputs:
                pixel_values = prompt_inputs['pixel_values']

                if isinstance(pixel_values, torch.Tensor):
                    if pixel_values.dim() >= 2 and pixel_values.shape[0] == 1:
                        pixel_values = pixel_values.squeeze(0)
                    output['pixel_values'] = pixel_values
                elif isinstance(pixel_values, list) and len(pixel_values) > 0:
                    if isinstance(pixel_values[0], torch.Tensor):
                        output['pixel_values'] = pixel_values[0] if len(pixel_values) == 1 else torch.cat(pixel_values, dim=0)

            for key in ['input_ids', 'attention_mask', 'labels', 'token_type_ids']:
                if key in output and not isinstance(output[key], torch.Tensor):
                    print(f"ERROR: {key} is not a tensor: {type(output[key])}")
                    return None

            return output

        except Exception as e:
            print(f"Error processing example: {e}")
            import traceback
            traceback.print_exc()
            return None

    print("Tokenizing datasets...")
    print("Processing training data...")
    tokenized_train_dataset = train_dataset.map(
        tokenize_vqa,
        remove_columns=train_dataset.column_names,
        num_proc=1,
        desc="Processing training data",
        load_from_cache_file=False
    )

    print("Filtering valid training samples...")
    tokenized_train_dataset = tokenized_train_dataset.filter(
        lambda x: x is not None and 'input_ids' in x and 'labels' in x,
        desc="Filtering training data"
    )

    print("Processing validation data...")
    tokenized_val_dataset = val_dataset.map(
        tokenize_vqa,
        remove_columns=val_dataset.column_names,
        num_proc=1,
        desc="Processing validation data",
        load_from_cache_file=False
    )

    print("Filtering valid validation samples...")
    tokenized_val_dataset = tokenized_val_dataset.filter(
        lambda x: x is not None and 'input_ids' in x and 'labels' in x,
        desc="Filtering validation data"
    )

    print("Processing test data (held out from validation)...")
    tokenized_test_dataset = test_dataset.map(
        tokenize_vqa,
        remove_columns=test_dataset.column_names,
        num_proc=1,
        desc="Tokenizing test",
        load_from_cache_file=False,
    )

    print("Filtering valid test samples...")
    tokenized_test_dataset = tokenized_test_dataset.filter(
        lambda x: x is not None and 'input_ids' in x and 'labels' in x,
        desc="Filtering test data"
    )

    train_len = len(tokenized_train_dataset)
    val_len = len(tokenized_val_dataset)
    test_len = len(tokenized_test_dataset)

    if train_len == 0:
        raise ValueError(
            "No training samples after processing. "
            "Possible causes: images not loading correctly, dataset format incompatible, "
            "or all examples failed validation."
        )

    print(f"\nSuccessfully processed datasets:")
    print(f"Training samples: {train_len}")
    print(f"Validation samples: {val_len}")
    print(f"Test samples: {test_len}")

    eff_batch = config.batch_size * config.gradient_accumulation_steps
    total_steps = math.ceil(train_len / eff_batch) * config.num_epochs

    print(f"\nTraining Configuration:")
    print(f"Total training steps: {total_steps}")
    print(f"Effective batch size: {eff_batch}")
    print(f"Steps per epoch: {math.ceil(train_len / eff_batch)}")

    training_args = Seq2SeqTrainingArguments(
        output_dir="./results_refcoco_lora_gemma3",
        per_device_train_batch_size=config.batch_size,
        per_device_eval_batch_size=config.batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        num_train_epochs=config.num_epochs,
        learning_rate=config.learning_rate,
        bf16=(config.precision == "bf16"),
        fp16=(config.precision == "fp16"),
        max_grad_norm=1.5,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        dataloader_num_workers=config.num_workers,
        dataloader_pin_memory=config.pin_memory,
        dataloader_drop_last=config.drop_last,
        remove_unused_columns=False,
        report_to="none",
        save_strategy="epoch",
        eval_strategy="epoch",
        save_total_limit=2,
        # FIX: logging_steps=50 meant train loss was essentially never
        # logged on this ~96-step run (only 2 log points total). And
        # warmup_steps=100 exceeded the entire run (96 steps with
        # grad_accum=1) or now, with grad_accum=4, the ~24 total optimizer
        # steps — meaning LR was still ramping up for the *entire* run and
        # never got to decay. 2 warmup steps (~10% of ~24 total optimizer
        # steps) is more appropriate for a run this short.
        logging_steps=2,
        warmup_steps=2,
        optim="adamw_torch",
        load_best_model_at_end=True,
        # FIX: teacher-forced eval loss can keep improving even as
        # free-running generation quality collapses (the model learns to
        # exploit EOS being an easy, frequent target under teacher forcing,
        # while at actual generation time — with no teacher forcing — it
        # starts emitting EOS immediately). Selecting "best" by loss alone
        # can therefore pick a checkpoint that outputs empty captions.
        # ROUGE-L is computed on the model's actual free-running
        # generations, so it directly reflects what we care about.
        metric_for_best_model="rouge_l",
        greater_is_better=True,
        prediction_loss_only=False,  # Need full predictions for metrics
        predict_with_generate=True,  # Use generation for evaluation
        generation_max_length=config.max_length,
    )

    def collate_fn(batch):
        """Custom collator for Gemma 3 inputs with proper batching."""
        batch = [b for b in batch if b is not None and isinstance(b, dict) and 'input_ids' in b]

        if len(batch) == 0:
            print("WARNING: Empty batch after filtering - creating dummy batch")
            dummy_input_ids = torch.zeros((1, config.max_length), dtype=torch.long)
            dummy_attention_mask = torch.zeros((1, config.max_length), dtype=torch.long)
            dummy_labels = torch.full((1, config.max_length), config.ignore_index, dtype=torch.long)
            dummy_token_type_ids = torch.zeros((1, config.max_length), dtype=torch.long)

            return {
                'input_ids': dummy_input_ids,
                'attention_mask': dummy_attention_mask,
                'labels': dummy_labels,
                'token_type_ids': dummy_token_type_ids,
            }

        try:
            for i, item in enumerate(batch):
                for key in ['input_ids', 'attention_mask', 'labels', 'token_type_ids']:
                    if key in item and not isinstance(item[key], torch.Tensor):
                        batch[i][key] = torch.tensor(item[key], dtype=torch.long)

            input_ids = torch.nn.utils.rnn.pad_sequence(
                [b['input_ids'] for b in batch],
                batch_first=True,
                padding_value=processor.tokenizer.pad_token_id
            )

            attention_mask = torch.nn.utils.rnn.pad_sequence(
                [b['attention_mask'] for b in batch],
                batch_first=True,
                padding_value=0
            )

            labels = torch.nn.utils.rnn.pad_sequence(
                [b['labels'] for b in batch],
                batch_first=True,
                padding_value=config.ignore_index
            )

            output = {
                'input_ids': input_ids,
                'attention_mask': attention_mask,
                'labels': labels,
            }

            if 'token_type_ids' in batch[0]:
                token_type_ids_list = [
                    b['token_type_ids'] for b in batch
                    if 'token_type_ids' in b and isinstance(b['token_type_ids'], torch.Tensor)
                ]
                if token_type_ids_list:
                    token_type_ids = torch.nn.utils.rnn.pad_sequence(
                        token_type_ids_list,
                        batch_first=True,
                        padding_value=0
                    )
                    output['token_type_ids'] = token_type_ids

            if 'pixel_values' in batch[0] and batch[0]['pixel_values'] is not None:
                pixel_values_list = []

                for b in batch:
                    pv = b['pixel_values']

                    if isinstance(pv, list):
                        if len(pv) > 0 and isinstance(pv[0], torch.Tensor):
                            pv = torch.stack(pv) if len(pv) > 1 else pv[0]
                        elif len(pv) > 0 and isinstance(pv[0], list):
                            pv = torch.tensor(pv)
                        else:
                            continue

                    if isinstance(pv, torch.Tensor):
                        while pv.dim() > 0 and pv.shape[0] == 1:
                            pv = pv.squeeze(0)
                        pixel_values_list.append(pv)

                if len(pixel_values_list) > 0:
                    try:
                        output['pixel_values'] = torch.stack(pixel_values_list, dim=0)
                    except RuntimeError:
                        max_shapes = [max(pv.shape[i] for pv in pixel_values_list) for i in range(pixel_values_list[0].dim())]
                        padded_pvs = []
                        for pv in pixel_values_list:
                            if list(pv.shape) != max_shapes:
                                padding = []
                                for i in range(len(max_shapes) - 1, -1, -1):
                                    padding.extend([0, max_shapes[i] - pv.shape[i]])
                                pv = torch.nn.functional.pad(pv, padding)
                            padded_pvs.append(pv)
                        output['pixel_values'] = torch.stack(padded_pvs, dim=0)

            # Pass the raw per-example human answers through untouched
            # (list of lists of strings) — needed for the official VQA
            # soft-accuracy metric. This is popped out again in
            # LoRATrainer.prediction_step before the tensors go to the model.
            output['vqa_answers'] = [b.get('vqa_answers', []) for b in batch]

            # FIX: carry prompt_length through so eval-time generation can
            # trim the answer back out of input_ids before calling
            # model.generate() (see LoRATrainer.prediction_step).
            if 'prompt_length' in batch[0]:
                output['prompt_length'] = torch.tensor(
                    [b.get('prompt_length', 0) for b in batch], dtype=torch.long
                )

            return output

        except Exception as e:
            print(f"ERROR in collate_fn: {e}")
            import traceback
            traceback.print_exc()

            dummy_input_ids = torch.zeros((1, config.max_length), dtype=torch.long)
            dummy_attention_mask = torch.zeros((1, config.max_length), dtype=torch.long)
            dummy_labels = torch.full((1, config.max_length), config.ignore_index, dtype=torch.long)
            dummy_token_type_ids = torch.zeros((1, config.max_length), dtype=torch.long)

            return {
                'input_ids': dummy_input_ids,
                'attention_mask': dummy_attention_mask,
                'labels': dummy_labels,
                'token_type_ids': dummy_token_type_ids,
            }

    def make_compute_metrics(processor, eval_answers_buffer):
        import re
        import numpy as np
        from rouge_score import rouge_scorer
        from nltk.translate.meteor_score import meteor_score
        from bert_score import score as bert_score
        from collections import defaultdict
        import nltk
        nltk.download('wordnet', quiet=True)
        nltk.download('omw-1.4', quiet=True)

        # PATCH FOR MOVERSCORE COMPATIBILITY
        import transformers.tokenization_utils_base as tub
        if not hasattr(tub.PreTrainedTokenizerBase, '_moverscore_patched'):
            original_init = tub.PreTrainedTokenizerBase.__init__
            def patched_init(self, *args, **kwargs):
                original_init(self, *args, **kwargs)
                if not hasattr(self, 'max_len'):
                    self.max_len = self.model_max_length
            tub.PreTrainedTokenizerBase.__init__ = patched_init
            tub.PreTrainedTokenizerBase._moverscore_patched = True
            print("✓ Applied moverscore compatibility patch")

        # Now import moverscore
        moverscore_available = False
        try:
            from moverscore_v2 import get_idf_dict, word_mover_score
            moverscore_available = True
            print("✓ MoverScore loaded successfully")
        except ImportError as e:
            print(f"✗ MoverScore not available: {e}")

        scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

        def extract_answer(text):
            """Clean Gemma output safely without truncating content."""

            if text is None:
                return ""

            # Remove special tokens
            text = text.replace("<bos>", "")
            text = text.replace("<eos>", "")
            text = text.replace("<start_of_turn>", "")
            text = text.replace("<end_of_turn>", "")

            # Remove leading 'model' tag if present
            text = re.sub(r"^\s*model\s*", "", text, flags=re.IGNORECASE)

            # Normalize whitespace
            text = re.sub(r"\s+", " ", text).strip()

            return text

        # Official VQA eval answer processing (from the VQA project's
        # vqaEval.py), used so exact-match comparisons for accuracy (and
        # the text-similarity metrics below) aren't broken by trailing
        # punctuation, articles, or number-word vs. digit mismatches.
        _vqa_contractions = {
            "aint": "ain't", "arent": "aren't", "cant": "can't", "couldve": "could've",
            "couldnt": "couldn't", "didnt": "didn't", "doesnt": "doesn't", "dont": "don't",
            "hadnt": "hadn't", "hasnt": "hasn't", "havent": "haven't", "hed": "he'd",
            "hes": "he's", "im": "i'm", "ive": "i've", "isnt": "isn't", "its": "it's",
            "lets": "let's", "shant": "shan't", "shes": "she's", "shouldve": "should've",
            "shouldnt": "shouldn't", "thats": "that's", "theres": "there's", "theyd": "they'd",
            "theyre": "they're", "theyve": "they've", "wasnt": "wasn't", "wed": "we'd",
            "were": "we're", "weve": "we've", "werent": "weren't", "whats": "what's",
            "wheres": "where's", "whos": "who's", "wont": "won't", "wouldve": "would've",
            "wouldnt": "wouldn't", "youd": "you'd", "youre": "you're", "youve": "you've",
        }
        _vqa_number_words = {
            "none": "0", "zero": "0", "one": "1", "two": "2", "three": "3", "four": "4",
            "five": "5", "six": "6", "seven": "7", "eight": "8", "nine": "9", "ten": "10",
        }
        _vqa_articles = {"a", "an", "the"}
        _vqa_punct = [';', '/', '[', ']', '"', '{', '}', '(', ')', '=', '+', '\\',
                      '_', '-', '>', '<', '@', '`', ',', '?', '!', '.', ':']

        def normalize(s):
            """Official VQA-style answer normalization."""
            s = s.lower().strip()
            # Remove punctuation entirely (rather than spacing it out),
            # so "brown." normalizes to "brown", not "brown .".
            for p in _vqa_punct:
                s = s.replace(p, ' ')
            s = re.sub(r"\s+", " ", s).strip()

            words = []
            for word in s.split():
                word = _vqa_number_words.get(word, word)
                if word in _vqa_articles:
                    continue
                word = _vqa_contractions.get(word, word)
                words.append(word)
            return ' '.join(words).strip()

        def compute_vqa_metrics(p: EvalPrediction):
            pred_ids = np.asarray(p.predictions, dtype=np.int64)
            labels = np.asarray(p.label_ids, dtype=np.int64)

            vocab_size = processor.tokenizer.vocab_size
            pad_token_id = processor.tokenizer.pad_token_id

            pred_ids = np.clip(pred_ids, 0, vocab_size - 1)
            labels[labels < 0] = pad_token_id
            labels = np.clip(labels, 0, vocab_size - 1)

            try:
                pred_texts_raw = processor.batch_decode(pred_ids, skip_special_tokens=True)
                label_texts = processor.batch_decode(labels, skip_special_tokens=True)
            except Exception as e:
                print(f"[Metrics] Decode error: {e}")
                return {"rouge_l": 0.0, "meteor": 0.0, "bertscore_f1": 0.0, "moverscore": 0.0, "accuracy": 0.0}

            pred_texts = [extract_answer(t) for t in pred_texts_raw]
            pred_norm = [normalize(t) for t in pred_texts]
            label_norm = [normalize(t) for t in label_texts]

            for i in range(min(3, len(pred_norm))):
                print(f"\n[Sample {i+1}]")
                print(f"Generated: {pred_texts[i][:100]}")
                print(f"Reference: {label_texts[i][:100]}")

            # FIX (root of the "half-baked scores" issue): everything below
            # used to be scored against `label_norm`, which is decoded from
            # the `labels` tensor — and `labels` only ever encodes
            # answers_list[0] (see tokenize_vqa), i.e. ONE arbitrary human
            # reference out of the 2-4 RefCOCO collects per region. A
            # correct prediction that happens to match reference #2 or #3
            # instead of #1 scored as a near-total miss on ROUGE-L/METEOR.
            #
            # lmms-eval's own aggregation for this exact dataset
            # (refcoco_aggregation_result) instead feeds ALL of
            # doc["answer"] in as separate reference captions for the same
            # image_id and lets pycocoevalcap's scorers take the best
            # match across the full reference set. We reproduce that here
            # using eval_answers_buffer (already populated by
            # LoRATrainer.prediction_step with the raw 2-4 answers per
            # example) instead of the single-answer label_norm.
            raw_answers_list = eval_answers_buffer.get("answers", [])
            use_multi_ref = len(raw_answers_list) == len(pred_norm)

            if use_multi_ref:
                norm_refs_per_example = [
                    [normalize(a) for a in raw_answers if a] or [l]
                    for raw_answers, l in zip(raw_answers_list, label_norm)
                ]
            else:
                print(
                    f"[Metrics] WARNING: buffered answers ({len(raw_answers_list)}) "
                    f"!= predictions ({len(pred_norm)}); falling back to "
                    f"single-reference scoring against the training answer."
                )
                norm_refs_per_example = [[l] for l in label_norm]

            valid_pairs = [
                (p, refs) for p, refs in zip(pred_norm, norm_refs_per_example)
                if p and refs and len(p.split()) > 0
            ]

            # ROUGE-L — max over this example's reference set.
            rouge_ls = []
            for p, refs in valid_pairs:
                best = max(
                    (scorer.score(r, p)['rougeL'].fmeasure for r in refs if r),
                    default=None,
                )
                if best is not None:
                    rouge_ls.append(best)
            rouge_l = np.mean(rouge_ls) if rouge_ls else 0.0

            # METEOR — nltk's meteor_score natively supports multiple
            # references and takes the max internally, so just pass the
            # full reference set instead of a single one.
            meteors = []
            for p, refs in valid_pairs:
                ref_token_lists = [r.split() for r in refs if r]
                if ref_token_lists:
                    meteors.append(meteor_score(ref_token_lists, p.split()))
            meteor = np.mean(meteors) if meteors else 0.0

            # BERTscore / MoverScore need one (pred, ref) string pair per
            # call, so we flatten pred-vs-each-of-its-refs into parallel
            # lists, score everything in one batched call, then take the
            # max back per original example.
            flat_preds, flat_refs, group_idx = [], [], []
            for i, (p, refs) in enumerate(valid_pairs):
                for r in refs:
                    if r:
                        flat_preds.append(p)
                        flat_refs.append(r)
                        group_idx.append(i)

            def _max_per_group(flat_scores, n_groups):
                best = [None] * n_groups
                for idx, s in zip(group_idx, flat_scores):
                    if best[idx] is None or s > best[idx]:
                        best[idx] = s
                return [b for b in best if b is not None]

            # BERTscore
            if flat_preds and flat_refs:
                try:
                    P, R, F1 = bert_score(
                        flat_preds,
                        flat_refs,
                        lang="en",
                        model_type="bert-base-uncased",
                        rescale_with_baseline=False
                    )
                    per_ex_best = _max_per_group(F1.tolist(), len(valid_pairs))
                    bertscore_f1 = float(np.mean(per_ex_best)) if per_ex_best else 0.0
                except Exception as e:
                    print(f"[Metrics] BERTScore error: {e}")
                    bertscore_f1 = 0.0
            else:
                bertscore_f1 = 0.0

            # Accuracy — FIX: the original code used the official VQA
            # leave-one-out soft-accuracy formula. That formula is designed
            # for VQA's setting (short, closed-set answers like "umpire",
            # ~10 annotators) and doesn't transfer well to RefCOCO's setting
            # (free-form referring expressions, only 2-4 annotators):
            #   1) The "/3" threshold caps max achievable accuracy below 1.0
            #      even for a perfect prediction with only 2-4 references
            #      (e.g. ceiling ≈ 0.67 with 3 refs).
            #   2) It requires an EXACT string match. A correct caption like
            #      "this is an umpire" will never exactly equal the
            #      reference "umpire", so accuracy stays near 0 regardless
            #      of how good the model actually is. This is why standard
            #      RefCOCO generation papers report METEOR/CIDEr/ROUGE-L/
            #      BLEU rather than "accuracy" at all — free-form text
            #      doesn't have a single correct string to match against.
            #
            # Fix: use a relaxed containment match instead — a prediction
            # counts as correct for an example if it exactly matches, or
            # contains, or is contained by, any one of that example's human
            # reference expressions (after normalization). This still
            # requires real word-level overlap with a genuine reference (so
            # it isn't free credit), but no longer structurally punishes a
            # correct model for writing "this is an umpire" instead of
            # exactly "umpire", and isn't capped by a VQA-specific ceiling.
            # Treat this as a coarse correct/incorrect signal — ROUGE-L /
            # METEOR / BERTScore / MoverScore remain the primary metrics
            # for judging caption quality.
            # (raw_answers_list was already fetched above for the
            # multi-reference NLG metrics — reused here as-is.)

            print(f"\n[Accuracy Debug] buffered answer-sets: {len(raw_answers_list)} | predictions: {len(pred_norm)}")
            for i in range(min(3, len(raw_answers_list), len(pred_norm))):
                print(f"[Accuracy Debug] Sample {i+1}")
                print(f"  Prediction (normalized): {pred_norm[i]!r}")
                print(f"  Raw human answers       : {raw_answers_list[i]}")
                print(f"  Normalized human answers: {[normalize(a) for a in raw_answers_list[i] if a]}")

            def relaxed_match(pred, ref):
                if not pred or not ref:
                    return False
                if pred == ref:
                    return True
                # Require at least a few characters of real overlap so a
                # single short common word (e.g. "the") can't trivially
                # match — both strings are already normalized (articles
                # stripped, lowercased) at this point.
                if len(ref) >= 3 and ref in pred:
                    return True
                if len(pred) >= 3 and pred in ref:
                    return True
                return False

            if not use_multi_ref:
                # Warning already printed above when building
                # norm_refs_per_example — just apply the same fallback here.
                matches = [1.0 if relaxed_match(p, l) else 0.0 for p, l in zip(pred_norm, label_norm) if l]
                accuracy = float(np.mean(matches)) if matches else 0.0
            else:
                per_question_accs = []
                for pred, raw_answers in zip(pred_norm, raw_answers_list):
                    norm_answers = [normalize(a) for a in raw_answers if a]
                    if not norm_answers:
                        continue
                    is_correct = any(relaxed_match(pred, ref) for ref in norm_answers)
                    per_question_accs.append(1.0 if is_correct else 0.0)
                accuracy = float(np.mean(per_question_accs)) if per_question_accs else 0.0



            # MoverScore — same flatten-then-max-per-example approach as
            # BERTScore above, using flat_preds/flat_refs/group_idx built
            # earlier from the FULL reference set (not just answers_list[0]).
            moverscore_val = 0.0
            if moverscore_available and flat_preds and flat_refs:
                try:
                    from moverscore_v2 import get_idf_dict, word_mover_score

                    idf_dict_ref = get_idf_dict(flat_refs)
                    idf_dict_hyp = get_idf_dict(flat_preds)

                    scores = word_mover_score(
                        flat_refs,
                        flat_preds,
                        idf_dict_ref,
                        idf_dict_hyp,
                        stop_words=[],
                        n_gram=1,
                        remove_subwords=True
                    )

                    if scores and len(scores) > 0:
                        per_ex_best = _max_per_group(scores, len(valid_pairs))
                        moverscore_val = float(np.mean(per_ex_best)) if per_ex_best else 0.0
                        print(f"[MoverScore] Score: {moverscore_val:.4f}")

                except Exception as e:
                    print(f"[MoverScore] Error: {e}")
                    import traceback
                    traceback.print_exc()

            return {
                "rouge_l": float(rouge_l),
                "meteor": float(meteor),
                "bertscore_f1": float(bertscore_f1),
                "moverscore": float(moverscore_val),
                "accuracy": accuracy,
            }

        return compute_vqa_metrics

    # Custom Trainer with VRAM clearing before evaluation
    class LoRATrainer(Seq2SeqTrainer):
        """Seq2SeqTrainer subclass that clears VRAM before evaluation."""

        def __init__(self, processor, eval_answers_buffer, gen_config, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self.processor = processor
            # Mutable dict shared with make_compute_metrics(), used to pass
            # the raw per-question human answers (popped out of each batch
            # below) through to compute_metrics for the official VQA
            # soft-accuracy formula, since EvalPrediction only carries
            # decoded predictions/labels.
            self.eval_answers_buffer = eval_answers_buffer
            self.gen_config = gen_config  # LoRAConfig instance

        def evaluate(self, *args, **kwargs):
            print("\nClearing VRAM before evaluation...")
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            self.eval_answers_buffer["answers"] = []
            return super().evaluate(*args, **kwargs)

        def prediction_step(
            self,
            model: torch.nn.Module,
            inputs: dict,
            prediction_loss_only: bool,
            ignore_keys: Optional[list] = None,
        ):
            """Memory-efficient prediction step from GRIT."""
            has_labels = "labels" in inputs
            if has_labels:
                labels = inputs.pop("labels")
            else:
                labels = None

            # Pull the raw human-answer lists out before anything touches
            # the model — they're plain Python lists, not tensors, and
            # model.generate()/forward() would choke on an unexpected kwarg.
            vqa_answers_batch = inputs.pop("vqa_answers", None)
            if vqa_answers_batch is not None:
                self.eval_answers_buffer["answers"].extend(vqa_answers_batch)

            # FIX: pull prompt_length out too — it's metadata for us, not a
            # real model input.
            prompt_lengths = inputs.pop("prompt_length", None)

            input_ids = inputs["input_ids"]
            input_len = input_ids.shape[1]

            # FIX: build a PROMPT-ONLY copy of the inputs for generation.
            # `inputs["input_ids"]` (built by tokenize_vqa) is
            # [prompt][answer][eos] for every split, train and eval alike —
            # correct for training's teacher-forced loss, but wrong for
            # calling model.generate(), which must never see the answer.
            # Feeding the full sequence into generate() means the model is
            # handed the ground-truth caption as context and asked to
            # "continue" past it — producing meaningless output (often
            # empty, or leftover chat-template tokens that skip_special_
            # tokens then strips to "") instead of an actual caption.
            if prompt_lengths is not None:
                attn = inputs["attention_mask"]
                pad_id = self.processor.tokenizer.pad_token_id
                trimmed_ids = [input_ids[i, : int(prompt_lengths[i])] for i in range(input_ids.shape[0])]
                trimmed_mask = [attn[i, : int(prompt_lengths[i])] for i in range(attn.shape[0])]
                max_p = max(t.shape[0] for t in trimmed_ids)

                # Re-pad on the LEFT (matches processor.padding_side="left",
                # which the model/generate expects for batched decoding).
                gen_input_ids = torch.full(
                    (len(trimmed_ids), max_p), pad_id, dtype=input_ids.dtype, device=input_ids.device
                )
                gen_attention_mask = torch.zeros(
                    (len(trimmed_mask), max_p), dtype=attn.dtype, device=attn.device
                )
                for i, (ids_i, mask_i) in enumerate(zip(trimmed_ids, trimmed_mask)):
                    gen_input_ids[i, max_p - ids_i.shape[0]:] = ids_i
                    gen_attention_mask[i, max_p - mask_i.shape[0]:] = mask_i

                gen_inputs = dict(inputs)
                gen_inputs["input_ids"] = gen_input_ids
                gen_inputs["attention_mask"] = gen_attention_mask

                if "token_type_ids" in inputs:
                    tt = inputs["token_type_ids"]
                    trimmed_tt = [tt[i, : int(prompt_lengths[i])] for i in range(tt.shape[0])]
                    gen_tt = torch.zeros((len(trimmed_tt), max_p), dtype=tt.dtype, device=tt.device)
                    for i, tt_i in enumerate(trimmed_tt):
                        gen_tt[i, max_p - tt_i.shape[0]:] = tt_i
                    gen_inputs["token_type_ids"] = gen_tt
                # pixel_values is per-example (whole image), unaffected by
                # sequence-length trimming — carried over unchanged via dict(inputs).

                gen_input_len = gen_input_ids.shape[1]
            else:
                # No prompt_length metadata available (e.g. an older cached
                # dataset) — fall back to the old (unsafe) behavior rather
                # than crashing, but this should not happen with datasets
                # produced by the current tokenize_vqa.
                gen_inputs = inputs
                gen_input_len = input_len

            with torch.no_grad():
                if prediction_loss_only:
                    outputs = model(**inputs)
                    loss = outputs.loss if has_labels else None
                    return (loss, None, None)

                # Generate from the PROMPT ONLY (see gen_inputs above).
                # FIX: capped max_new_tokens (was 512) — reference captions
                # are a handful of words, so a small cap is enough once the
                # model has learned to stop, and keeps eval fast in the
                # meantime instead of always running to 512 tokens.
                # FIX: min_new_tokens forces the model to emit at least a
                # few real tokens before EOS is allowed. With only 32
                # training examples, the model sometimes hasn't learned
                # enough about a given (unseen) image to be confident, and
                # its cheapest option is to emit EOS immediately — an
                # empty caption that scores a guaranteed 0.0 on every
                # metric and tells you nothing about whether the
                # underlying captioning is "close but wrong" or "totally
                # lost". Forcing a few tokens out gives you an actual
                # (possibly wrong) caption to evaluate instead.
                generated_tokens = model.generate(
                    **gen_inputs,
                    max_new_tokens=self.gen_config.generation_max_new_tokens,
                    min_new_tokens=2,
                    do_sample=False,
                    pad_token_id=self.processor.tokenizer.pad_token_id,
                    eos_token_id=self.processor.tokenizer.eos_token_id,
                )

                # Slice to generated part only (answer)
                generated_tokens = generated_tokens[:, gen_input_len:]

                # Loss (full inputs for compute_loss) — uses the FULL
                # [prompt][answer] sequence, since teacher-forced loss is
                # supposed to see the real answer.
                loss = None
                if has_labels:
                    inputs_copy = inputs.copy()
                    inputs_copy["labels"] = labels
                    loss = self.compute_loss(model, inputs_copy).detach()

            # Labels processing
            if labels is not None:
                if labels.dim() == 3:
                    labels = torch.argmax(labels, dim=-1)
                elif labels.dim() == 1:
                    batch_size = input_ids.shape[0]
                    labels = labels.view(batch_size, -1)

            return loss, generated_tokens, labels

    # Shared mutable buffer: LoRATrainer.prediction_step fills it with the
    # raw per-question human answers for each eval batch; compute_metrics
    # reads it to compute the official VQA soft-accuracy formula.
    eval_answers_buffer = {"answers": []}

    trainer = LoRATrainer(
        processor=processor,
        eval_answers_buffer=eval_answers_buffer,
        gen_config=config,
        model=model,
        args=training_args,
        train_dataset=tokenized_train_dataset,
        eval_dataset=tokenized_val_dataset,
        data_collator=collate_fn,
        compute_metrics=make_compute_metrics(processor, eval_answers_buffer),
    )

    print("\n" + "="*80)
    print("STARTING LORA-ONLY TRAINING FOR GEMMA 3-4B-IT MODEL ON REFCOCO")
    print("="*80)
    print(f"Model: {config.model_id}")
    print(f"Dataset: {config.dataset_name}")
    print(f"LoRA rank: {config.lora_rank}")
    print(f"Batch size: {config.batch_size} x {config.gradient_accumulation_steps} = {eff_batch}")
    print(f"Learning rate: {config.learning_rate}")
    print(f"Precision: {config.precision}")
    print(f"Training samples: {train_len}")
    print(f"Validation samples: {val_len}")
    print(f"Test samples: {test_len}")
    print("="*80 + "\n")

    try:
        print("Beginning training...")
        trainer.train()

        print("\n" + "="*80)
        print("FINAL TEST EVALUATION")
        print("="*80)

        test_results = trainer.evaluate(eval_dataset=tokenized_test_dataset)

        print(f"Test Loss          : {test_results['eval_loss']:.5f}")
        print(f"Test ROUGE-L       : {test_results['eval_rouge_l']:.4f}")
        print(f"Test METEOR        : {test_results['eval_meteor']:.4f}")
        print(f"Test BERTscore F1  : {test_results['eval_bertscore_f1']:.4f}")
        print(f"Test MoverScore    : {test_results['eval_moverscore']:.4f}")
        print(f"Test Accuracy      : {test_results['eval_accuracy']:.4f}")

        print("\n" + "="*80)
        print("Saving final model...")
        output_dir = "./final_refcoco_lora_gemma3_model"
        model.save_pretrained(output_dir)
        processor.save_pretrained(output_dir)
        print(f"Model saved to {output_dir}")

        print("\n" + "="*80)
        print("TRAINING COMPLETED SUCCESSFULLY")
        print("="*80 + "\n")

    except KeyboardInterrupt:
        print("\n" + "="*80)
        print("TRAINING INTERRUPTED BY USER")
        print("="*80)
        print("Saving checkpoint...")
        try:
            model.save_pretrained("./interrupted_checkpoint_lora_only_gemma3")
            processor.save_pretrained("./interrupted_checkpoint_lora_only_gemma3")
            print("Checkpoint saved to ./interrupted_checkpoint_lora_only_gemma3")
        except Exception as e:
            print(f"Failed to save checkpoint: {e}")

    except Exception as e:
        print("\n" + "="*80)
        print("TRAINING FAILED")
        print("="*80)
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()
        raise

    finally:
        print("\nCleaning up...")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print("Cleanup complete")


if __name__ == "__main__":
    main()

Loading processor and model for Gemma 3-4B-IT (RefCOCO)...


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.61k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

... (64 outputs left out)

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[Accuracy Debug] buffered answer-sets: 4 | predictions: 4
[Accuracy Debug] Sample 1
  Prediction (normalized): 'orange segment'
  Raw human answers       : ['orange in bowl', 'top mid orange whole']
  Normalized human answers: ['orange in bowl', 'top mid orange whole']
[Accuracy Debug] Sample 2
  Prediction (normalized): 'stovetop and cookware'
  Raw human answers       : ['stove', 'stove top', 'stove top']
  Normalized human answers: ['stove', 'stove top', 'stove top']
[Accuracy Debug] Sample 3
  Prediction (normalized): 'crumb of donut'
  Raw human answers       : ['top edge, 3rd donut from right', 'portion of donut at top left of center']
  Normalized human answers: ['top edge 3rd donut from right', 'portion of donut at top left of center']
[MoverScore] Score: 0.5569


Training Loss,Validation Loss,Epoch,Rouge L,Meteor,Bertscore F1,Moverscore,Accuracy
6.697441,7.437994,3,0.274242,0.118103,0.608124,0.556855,0.250000


Test Loss          : 7.43799
Test ROUGE-L       : 0.2742
Test METEOR        : 0.1181
Test BERTscore F1  : 0.6081
Test MoverScore    : 0.5569
Test Accuracy      : 0.2500

Saving final model...
Model saved to ./final_refcoco_lora_gemma3_model

TRAINING COMPLETED SUCCESSFULLY


Cleaning up...
Cleanup complete
